In [62]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [139]:
books = pd.read_excel("C:/Users/jb24d625/OneDrive/Doktorat/CAS/Modul 6/book_recommender/archive (1)/Amazon_Books_Scraping/Books_df.xlsx")

In [140]:
print(books.head())

                                               Title               Author  \
0              The Complete Novel of Sherlock Holmes   Arthur Conan Doyle   
1  Black Holes (L) : The Reith Lectures [Paperbac...      Stephen Hawking   
2                                    The Kite Runner      Khaled Hosseini   
3  Greenlights: Raucous stories and outlaw wisdom...  Matthew McConaughey   
4  The Science of Storytelling: Why Stories Make ...           Will Storr   

                 Main Genre           Sub Genre            Type      Price  \
0  Arts, Film & Photography  Cinema & Broadcast       Paperback  â‚¹169.00   
1  Arts, Film & Photography  Cinema & Broadcast       Paperback   â‚¹99.00   
2  Arts, Film & Photography  Cinema & Broadcast  Kindle Edition  â‚¹175.75   
3  Arts, Film & Photography  Cinema & Broadcast       Paperback  â‚¹389.00   
4  Arts, Film & Photography  Cinema & Broadcast       Paperback  â‚¹348.16   

   Rating  No. of People rated  \
0     4.4                19923   


In [141]:
books

,Title,Author,Main Genre,Sub Genre,Type,Price,Rating,No. of People rated,URLs
0,The Complete Novel of Sherlock Holmes,Arthur Conan Doyle,"Arts, Film & Photography",Cinema & Broadcast,Paperback,â‚¹169.00,4.4,19923,https://www.amazon.in/Complete-Novels-Sherlock...
1,Black Holes (L) : The Reith Lectures [Paperbac...,Stephen Hawking,"Arts, Film & Photography",Cinema & Broadcast,Paperback,â‚¹99.00,4.5,7686,https://www.amazon.in/Black-Holes-Lectures-Ste...
2,The Kite Runner,Khaled Hosseini,"Arts, Film & Photography",Cinema & Broadcast,Kindle Edition,â‚¹175.75,4.6,50016,https://www.amazon.in/Kite-Runner-Khaled-Hosse...
3,Greenlights: Raucous stories and outlaw wisdom...,Matthew McConaughey,"Arts, Film & Photography",Cinema & Broadcast,Paperback,â‚¹389.00,4.6,32040,https://www.amazon.in/Greenlights-Raucous-stor...
4,The Science of Storytelling: Why Stories Make ...,Will Storr,"Arts, Film & Photography",Cinema & Broadcast,Paperback,â‚¹348.16,4.5,1707,https://www.amazon.in/Science-Storytelling-Wil...
...,...,...,...,...,...,...,...,...,...
7897,Insight Guides Poland (Travel Guide with Free ...,Insight Travel Guide,Travel,Travel & Holiday Guides,Paperback,"â‚¹1,326.00",4.7,16,https://www.amazon.in/Insight-Guides-Poland-Tr...
7898,Lonely Planet India 19 (Travel Guide),Anirban Mahapatra,Travel,Travel & Holiday Guides,Paperback,â‚¹850.00,4.4,187,https://www.amazon.in/Lonely-Planet-India-Trav...
7899,Eyewitness Travel Phrase Book French (EW Trave...,DK,Travel,Travel & Holiday Guides,Paperback,â‚¹307.00,4.5,168,https://www.amazon.in/Eyewitness-Travel-Phrase...
7900,Lonely Planet Australia (Travel Guide),Andrew Bain,Travel,Travel & Holiday Guides,Kindle Edition,"â‚¹1,814.50",4.7,267,https://www.amazon.in/Lonely-Planet-Australia-...


In [142]:
#remove Type, Price and URLs
books = books.drop(columns = "Type")
books = books.drop(columns = "Price")
books = books.drop(columns = "URLs")

#remove duplicates
books = books.drop_duplicates(subset='Title')

In [143]:
#rename columns for cleaner code
books.rename(columns={
    'Title': 'title',
    'Author': 'author',
    'Main Genre': 'main_genre',
    'Sub Genre': 'sub_genre',
    'Rating': 'average_rating',
    'No. of People rated': 'ratingsCount'
}, inplace=True)

In [144]:
print(books.columns)

Index(['title', 'author', 'main_genre', 'sub_genre', 'average_rating',
       'ratingsCount'],
      dtype='object')


In [145]:
#Check datatypes
books.dtypes

title              object
author             object
main_genre         object
sub_genre          object
average_rating    float64
ratingsCount        int64
dtype: object

In [146]:
#we want to build a simple TF-IDF recommendation system so we need strings for text columns instead of objects and combine them to one column
books['content'] = (
    books['title'].astype(str) + " " +
    books['author'].astype(str) + " " +
    books['main_genre'].astype(str) + " " +
    books['sub_genre'].astype(str)
)

In [147]:
books.dtypes

title              object
author             object
main_genre         object
sub_genre          object
average_rating    float64
ratingsCount        int64
content            object
dtype: object

In [148]:
#pandas often uses object as generic term, so we test whether string worked even though it does not say string
print(type(books['content'].iloc[0]))

<class 'str'>


In [149]:
#check missing data
books.isnull()

,title,author,main_genre,sub_genre,average_rating,ratingsCount,content
0,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...
7897,False,False,False,False,False,False,False
7898,False,False,False,False,False,False,False
7899,False,False,False,False,False,False,False
7900,False,False,False,False,False,False,False


TF-IDF

In [153]:
#TF is the term frequency which shows how often a word appears in the strings of the content column
#IDF is the inverse document frequency that shows how rare the word is across all books
#I use the stop words from the english language to remove useless words like the, and, is etc. 

vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(books['content'])



In [154]:
print("TF-IDF shape:", tfidf_matrix.shape)

TF-IDF shape: (5722, 12643)


In [155]:
#I use cosine similarity to compare every book with every other book. 1.0 = identical 0.0 = completely different
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [156]:
#map the titles to drop books that are duplicates
indices = pd.Series(books.index, index=books['title'])

In [157]:
#recommendation function, given a book name system returns 5 top similar books
#index given so that I can access the book and check the cosine similarity of this book with all other books
def recommend_books(title, top_n=5): 
    if title not in indices: 
        return f"Book '{title}' not found." 
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    book_indices = [i[0] for i in sim_scores]

    return books[['title', 'author', 'main_genre']].iloc[book_indices]

In [158]:
#test
book_name = "Fantastic Beasts and Where to Find Them: The Original Screenplay"
print("\nRecommendations:\n") 
print(recommend_books(book_name))


Recommendations:

                                                title        author  \
6   Fantastic Beasts: The Crimes of Grindelwald - ...  J.K. Rowling   
23  FANTASTIC BEASTS: THE SECRETS OF DUMBLEDORE ï¿...  J.K. Rowling   
9                                          Screenplay     Syd Field   
40                                               Will    Will Smith   
33  The Magic of MinaLima: Celebrating the Graphic...      MinaLima   

                  main_genre  
6   Arts, Film & Photography  
23  Arts, Film & Photography  
9   Arts, Film & Photography  
40  Arts, Film & Photography  
33  Arts, Film & Photography  


Embedded Vector Recommender with Ratings

In [159]:
#used to base the recommendation on semantic similarity instead of cosine similarity
#import additional library to convert text to embeddings
!pip install sentence-transformers

from sentence_transformers import SentenceTransformer

In [160]:
#load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(books['content'].tolist())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [161]:
#cosine similarity
cosine_sim = cosine_similarity(embeddings, embeddings)

In [162]:
#weighted rating for a hybrid recommender system
C = books['average_rating'].mean() 
m = books['ratingsCount'].quantile(0.75) 

def weighted_rating(x, m=m, C=C): 
    v = x['ratingsCount'] 
    R = x['average_rating'] 
    return (v / (v + m)) * R + (m / (v + m)) * C 
    
books['score'] = books.apply(weighted_rating, axis=1)

In [163]:
#first you find the index of the book you want
#then you get the similarity scores and sort them with the most similar books first
def hybrid_recommend(title, top_n=10):

    if title not in indices: 
        return f"books '{title}' not found." 

    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx])) 
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:50]
    book_indices = [i[0] for i in sim_scores]
    candidates = books.iloc[book_indices].copy()
    candidates['similarity'] = [i[1] for i in sim_scores]
    candidates['norm_score'] = candidates['score'] / candidates['score'].max()
    candidates['final_score'] = (
        0.7 * candidates['similarity'] +
        0.3 * candidates['norm_score']
    )
    candidates = candidates.sort_values('final_score', ascending=False)

    return candidates[['title', 'author', 'main_genre','average_rating', 'ratingsCount']].head(top_n)

In [164]:
#test
book_name = "Fantastic Beasts and Where to Find Them: The Original Screenplay"
print("\nRecommendations:\n") 
print(hybrid_recommend(book_name))


Recommendations:

                                                  title         author  \
6     Fantastic Beasts: The Crimes of Grindelwald - ...   J.K. Rowling   
23    FANTASTIC BEASTS: THE SECRETS OF DUMBLEDORE ï¿...   J.K. Rowling   
33    The Magic of MinaLima: Celebrating the Graphic...       MinaLima   
100   Harry Potter and the Cursed Child - Parts One ...   J.K. Rowling   
900             Harry Potter and the Chamber of Secrets   J.K. Rowling   
962                Harry Potter and the Deathly Hallows   J.K. Rowling   
897            Harry Potter and the Philosopher's Stone   J.K. Rowling   
961           Harry Potter and the Order of the Phoenix   J.K. Rowling   
7737          Harry Potter - The Illustrated Collection  J. K. Rowling   
903                 Harry Potter and the Goblet of Fire   J.K. Rowling   

                    main_genre  average_rating  ratingsCount  
6     Arts, Film & Photography             4.6         10152  
23    Arts, Film & Photography          